# 🎙️ Resemble Enhance — AI Audio Enhancement (GPU)

**Upute:**
1. `Runtime → Change runtime type → GPU (T4)` 
2. Pokreni sve ćelije redom
3. Uploadaj WAV kad te skripta zatraži
4. Preuzmi poboljšani fajl na kraju

## 1. Instalacija

In [ ]:
!pip install -q resemble-enhance torchaudio ffmpeg-normalize

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. Upload WAV fajla

In [ ]:
# Opcija A: Upload iz browsera
from google.colab import files
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
print(f"Uploadano: {INPUT_FILE}")

In [ ]:
# Opcija B: Ako je fajl prevelik za upload (>1GB), koristi Google Drive
# Odkomentiraj ove linije i zakomentiraj Opciju A gornju celiju

# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_FILE = "/content/drive/MyDrive/FINALNI_PODCAST_AUDIO.wav"  # <-- prilagodi putanju

## 3. AI Enhancement (GPU)

In [ ]:
import os
import gc
import time
import tempfile
import glob
import torch
import torchaudio
from resemble_enhance.enhancer.inference import enhance

# --- Parametri ---
LAMBD = 0.5            # Balans original/AI (0.0=originalniji, 1.0=vise AI)
SEGMENT_SECONDS = 120  # GPU ima dovoljno VRAM-a za vece segmente
OVERLAP_SECONDS = 2
OUTPUT_FILE = INPUT_FILE.replace(".wav", "_enhanced.wav").replace(".WAV", "_enhanced.wav")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")

info = torchaudio.info(INPUT_FILE)
sr = info.sample_rate
total_frames = info.num_frames
segment_frames = SEGMENT_SECONDS * sr
overlap_frames = OVERLAP_SECONDS * sr
hop_frames = segment_frames - overlap_frames

total_segments = max(1, (total_frames - overlap_frames + hop_frames - 1) // hop_frames)
print(f"Audio: {total_frames/sr:.0f}s @ {sr}Hz, {info.num_channels}ch")
print(f"Segmenti: {total_segments} x {SEGMENT_SECONDS}s")
print(f"Procjena: ~{total_segments * 0.5:.0f} min na T4 GPU")

tmpdir = tempfile.mkdtemp(prefix="enhance_")
segment_files = []
new_sr = None
t0 = time.time()

for i in range(total_segments):
    start = i * hop_frames
    num_frames = min(segment_frames, total_frames - start)

    dwav, seg_sr = torchaudio.load(INPUT_FILE, frame_offset=start, num_frames=num_frames)
    dwav = dwav.mean(0)  # Mono

    chunk_enhanced, new_sr = enhance(
        dwav, seg_sr, device,
        nfe=64, solver="midpoint", lambd=LAMBD, tau=0.5,
    )

    seg_path = os.path.join(tmpdir, f"seg_{i:04d}.wav")
    out = chunk_enhanced.unsqueeze(0).cpu() if chunk_enhanced.dim() == 1 else chunk_enhanced.cpu()
    torchaudio.save(seg_path, out, new_sr)
    segment_files.append(seg_path)

    del dwav, chunk_enhanced, out
    gc.collect()
    torch.cuda.empty_cache()

    elapsed = time.time() - t0
    per_seg = elapsed / (i + 1)
    remaining = per_seg * (total_segments - i - 1)
    print(f"  [{i+1}/{total_segments}] {start/sr:.0f}s-{(start+num_frames)/sr:.0f}s  OK  ({remaining/60:.0f} min preostalo)")

# Spoji segmente
print(f"\nSpajam {len(segment_files)} segmenata...")
overlap_enhanced = int(OVERLAP_SECONDS * new_sr)
half_overlap = overlap_enhanced // 2

pieces = []
for i, seg_path in enumerate(segment_files):
    seg, _ = torchaudio.load(seg_path)
    seg = seg[0]
    if len(segment_files) == 1:
        pieces.append(seg)
    elif i == 0:
        pieces.append(seg[:-half_overlap])
    elif i == len(segment_files) - 1:
        pieces.append(seg[half_overlap:])
    else:
        pieces.append(seg[half_overlap:-half_overlap])
    del seg

result = torch.cat(pieces).unsqueeze(0)
torchaudio.save(OUTPUT_FILE, result, new_sr)

# Cleanup
for f in glob.glob(os.path.join(tmpdir, "*.wav")):
    os.remove(f)
os.rmdir(tmpdir)

total_time = time.time() - t0
print(f"\n✅ Gotovo za {total_time/60:.1f} min!")
print(f"Spremljeno: {OUTPUT_FILE} ({result.shape[-1]/new_sr:.0f}s @ {new_sr}Hz)")

## 4. LUFS normalizacija (streaming standard)

In [ ]:
NORMALIZED_FILE = OUTPUT_FILE.replace("_enhanced.wav", "_final_premium.wav")

!ffmpeg-normalize "{OUTPUT_FILE}" \
  -t -16 \
  -tp -1.0 \
  -ar 48000 \
  -c:a pcm_s16le \
  -o "{NORMALIZED_FILE}" \
  -f

print(f"\n✅ LUFS normalizirano: {NORMALIZED_FILE}")

## 5. Preuzimanje

In [ ]:
# Opcija A: Direktan download u browser
from google.colab import files
files.download(NORMALIZED_FILE)

In [ ]:
# Opcija B: Kopiraj na Google Drive (bolje za velike fajlove)
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(NORMALIZED_FILE, "/content/drive/MyDrive/")
# print("Kopirano na Google Drive!")